# Weather Gold Layer — KPI Aggregation

Reads from `workspace.silver.weather_readings`, computes KPIs per city (latest reading upsert), derives the KPI columns `state_region`, `dew_point_c` (Magnus-Tetens dew point), `heat_index_c` (Rothfusz heat index, NULL below 26.7 °C) and `severity_score` (0–100 composite), and writes to `workspace.gold.weather_current_kpis`.

Uses `foreachBatch` + `DeltaTable.merge` for idempotent upserts.

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

CHECKPOINT_LOCATION = "/Volumes/workspace/landing/checkpoints/weather/gold"
GOLD_TABLE = "workspace.gold.weather_current_kpis"

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

# --- Derived KPI expressions (single source of truth for upsert + backfill) ---
STATE_REGION_EXPR = """
CASE
    WHEN city = 'Ahmedabad' THEN 'Gujarat'        WHEN city = 'Bengaluru' THEN 'Karnataka'
    WHEN city = 'Chennai'    THEN 'Tamil Nadu'    WHEN city = 'Delhi'     THEN 'Delhi (NCT)'
    WHEN city = 'Hyderabad'  THEN 'Telangana'     WHEN city = 'Jaipur'    THEN 'Rajasthan'
    WHEN city = 'Kochi'      THEN 'Kerala'        WHEN city = 'Kolkata'   THEN 'West Bengal'
    WHEN city = 'Mumbai'     THEN 'Maharashtra'   WHEN city = 'Pune'      THEN 'Maharashtra'
    WHEN city = 'Dubai'      THEN 'UAE (Dubai)'   WHEN city = 'London'    THEN 'UK (England)'
    WHEN city = 'New York'   THEN 'USA (New York)' WHEN city = 'Sydney'   THEN 'Australia (NSW)'
    WHEN city = 'Tokyo'      THEN 'Japan (Tokyo)' ELSE 'Other'
END"""

DEW_POINT_EXPR = """
ROUND(243.04 * (LN(relative_humidity_2m/100) + (17.625*temperature_2m)/(243.04+temperature_2m))
      / (17.625 - (LN(relative_humidity_2m/100) + (17.625*temperature_2m)/(243.04+temperature_2m))), 1)"""

HEAT_INDEX_EXPR = """
CASE
    WHEN temperature_2m >= 26.7 THEN
        ROUND(((-42.379
          + 2.04901523*((temperature_2m*9/5)+32) + 10.14333127*relative_humidity_2m
          - 0.22475541*((temperature_2m*9/5)+32)*relative_humidity_2m
          - 0.00683783*POWER(((temperature_2m*9/5)+32),2) - 0.05481717*POWER(relative_humidity_2m,2)
          + 0.00122874*POWER(((temperature_2m*9/5)+32),2)*relative_humidity_2m
          + 0.00085282*((temperature_2m*9/5)+32)*POWER(relative_humidity_2m,2)
          - 0.00000199*POWER(((temperature_2m*9/5)+32),2)*POWER(relative_humidity_2m,2)) - 32) * 5/9, 1)
    ELSE NULL
END"""

SEVERITY_SCORE_EXPR = """
ROUND(40*LEAST(wind_gusts_10m/60.0,1) + 30*LEAST(precipitation/10.0,1)
      + 20*(cloud_cover/100.0) + 10*LEAST(ABS(pressure_msl-1013)/25.0,1), 0)"""

# Idempotent schema evolution: add only the derived KPI columns still missing
if spark.catalog.tableExists(GOLD_TABLE):
    existing = set(spark.table(GOLD_TABLE).columns)
    missing = [c for c in [("state_region", "STRING"), ("dew_point_c", "DOUBLE"),
                           ("heat_index_c", "DOUBLE"), ("severity_score", "DOUBLE")]
               if c[0] not in existing]
    if missing:
        cols = ", ".join(f"{n} {t}" for n, t in missing)
        spark.sql(f"ALTER TABLE {GOLD_TABLE} ADD COLUMNS ({cols})")

In [0]:
silver = spark.readStream.table("workspace.silver.weather_readings")

def upsert_kpis(batch_df, batch_id):
    if batch_df.isEmpty():
        return
    from pyspark.sql.window import Window
    w = Window.partitionBy("city").orderBy(col("polled_at").desc())
    latest = (batch_df
        .withColumn("rn", row_number().over(w))
        .filter("rn = 1")
        .drop("rn")
        .select(
            "city", "latitude", "longitude", "elevation",
            "observation_time", "polled_at",
            "temperature_2m", "apparent_temperature", "relative_humidity_2m",
            "precipitation", "rain", "weather_code", "cloud_cover",
            "pressure_msl", "wind_speed_10m", "wind_direction_10m",
            "wind_gusts_10m", "is_day",
        ))

    # Derived KPI columns (expressions defined in the Config cell)
    latest = (latest
        .withColumn("state_region", expr(STATE_REGION_EXPR))
        .withColumn("dew_point_c", expr(DEW_POINT_EXPR))
        .withColumn("heat_index_c", expr(HEAT_INDEX_EXPR))
        .withColumn("severity_score", expr(SEVERITY_SCORE_EXPR)))

    if spark.catalog.tableExists(GOLD_TABLE):
        delta_table = DeltaTable.forName(spark, GOLD_TABLE)
        (delta_table.alias("t").merge(
            latest.alias("s"), "t.city = s.city")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
    else:
        latest.write.format("delta").saveAsTable(GOLD_TABLE)

query = (silver.writeStream
    .foreachBatch(upsert_kpis)
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .start())

query.awaitTermination()
print(f"✅ Gold stream completed — KPIs upserted into {GOLD_TABLE}")

In [0]:
# Idempotent backfill: recompute the derived KPIs for every existing gold row.
# Needed once after the columns were added (the streaming upsert only touches
# cities with new observations); safe to re-run anytime.
spark.sql(f"""
    UPDATE {GOLD_TABLE} SET
        state_region   = {STATE_REGION_EXPR},
        dew_point_c   = {DEW_POINT_EXPR},
        heat_index_c  = {HEAT_INDEX_EXPR},
        severity_score = {SEVERITY_SCORE_EXPR}
""")

print(f"✅ KPI backfill complete — {spark.table(GOLD_TABLE).count()} cities carry derived KPI values")

In [0]:
row_count = spark.table(GOLD_TABLE).count()
print(f"📊 Gold table {GOLD_TABLE}: {row_count} cities with current KPIs")

spark.sql(f"""
SELECT 
    city,
    ROUND(temperature_2m, 1) AS temp_c,
    ROUND(apparent_temperature, 1) AS feels_like_c,
    ROUND(relative_humidity_2m, 0) AS humidity_pct,
    ROUND(wind_speed_10m, 1) AS wind_kmh,
    ROUND(precipitation, 2) AS precip_mm,
    weather_code,
    CASE WHEN is_day = 1 THEN 'Day' ELSE 'Night' END AS day_night,
    observation_time,
    state_region, dew_point_c, heat_index_c, severity_score
FROM {GOLD_TABLE}
ORDER BY temp_c DESC
""").display()